[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab4_GraphRAG_KnowledgeGraphs_Student.ipynb)


# Lab 4: GraphRAG — Knowledge Graphs for Clinical Knowledge
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Vector RAG is great for 'what is the dose for cisplatin?' but fails for 'what are the relationships between staging, histology, and treatment?' GraphRAG extracts entities and relationships, builds a knowledge graph, and answers multi-hop clinical questions.

### Objective
- Extract clinical entities and relationships from WT guidelines
- Build a knowledge graph with NetworkX
- Visualize the graph
- Query with graph traversal
- Compare with vector RAG on multi-hop questions

---
## Setup

Install packages and set Colab secrets (`OPENAI_API_KEY`, `LLAMA_CLOUD_API_KEY`).

**What is GraphRAG?**  
Standard vector RAG retrieves *chunks of text* close to your query. GraphRAG instead extracts *entities* (drugs, stages, diseases) and *relationships* (TREATS, CAUSES) into a graph, then answers questions by traversing edges — no embeddings needed for the reasoning step.

**Lab flow:**
1. Define a typed clinical ontology (entities + relationship types)
2. Load WT guideline text → split into ~8 coherent chunks
3. LLM with `structured_output` extracts a `GraphExtraction` object per chunk
4. Assemble entities/relationships into a NetworkX `DiGraph`
5. Visualize interactively with pyvis
6. Answer multi-hop questions via graph traversal
7. Detect thematic clusters with Louvain community detection
8. Compare with vector RAG on the same question

Provide **`/content/wt.md`** (saved from Lab 1) or let the notebook load it from Google Drive.

In [ ]:
!pip install -q networkx matplotlib langchain langchain-text-splitters langchain-openai openai pyvis langchain-community llama-parse faiss-cpu tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('Loaded API keys from Colab secrets.')


## Cell 1 — Define entity and relationship schemas

GraphRAG requires a **typed ontology** — a controlled vocabulary the LLM must map text into.

**Entity types** (clinical nouns): `Disease`, `Stage`, `Drug`, `SideEffect`, `Procedure`, `Histology`, `Outcome`

**Relationship types** (directed edges): `TREATS`, `CAUSES`, `INDICATES`, `REQUIRES`, `ASSOCIATED_WITH`, `PART_OF`

Using `Literal[...]` in Pydantic + `.with_structured_output(GraphExtraction)` forces the LLM to output valid typed JSON rather than free text. This makes extraction **deterministic and parseable** — every entity snaps into a known category rather than being an arbitrary string.

In [ ]:
from pydantic import BaseModel
from typing import List, Literal

class Entity(BaseModel):
    name: str
    type: Literal["Disease", "Stage", "Drug", "SideEffect", "Procedure", "Histology", "Outcome"]
    description: str

class Relationship(BaseModel):
    source: str  # entity name
    target: str  # entity name
    type: Literal["TREATS", "CAUSES", "INDICATES", "REQUIRES", "ASSOCIATED_WITH", "PART_OF"]
    description: str

class GraphExtraction(BaseModel):
    entities: List[Entity]
    relationships: List[Relationship]

---
## Cell 2 — Load text chunks from WT.pdf

We work with a **small subset (~8 chunks)** for two reasons:
1. **Cost control** — each chunk triggers one LLM API call for graph extraction
2. **Coherence** — 1500-char chunks with 150-char overlap preserve complete guideline sections

The keyword filter (`stage`, `chemotherapy`, `vincristine`, etc.) targets the *treatment protocol* pages where entity density is highest. Processing the full document adds noise (patient-info pages, bibliography) without improving the graph.

Provide **`/content/wt.md`** from Lab 1 via Google Drive or PC upload.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

MD_PATH = '/content/wt.md'

# ── Option 1: Load WT_parsed.md from Google Drive (recommended) ──────────────────────────────────
if not os.path.exists(MD_PATH):
    try:
        from google.colab import drive
        import shutil
        drive.mount('/content/drive')
        shutil.copy('/content/drive/MyDrive/CCI_Session6/WT_parsed.md', MD_PATH)
        print(f"Loaded WT_parsed.md from Google Drive → {MD_PATH}")
    except Exception as _e:
        print(f"Drive unavailable: {_e}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────

# ── Option 2: Upload WT_parsed.md from your PC ───────────────────────────────────────────────────
# if not os.path.exists(MD_PATH):
#     from google.colab import files
#     import shutil
#     files.upload()  # pick WT_parsed.md
#     shutil.copy('WT_parsed.md', MD_PATH)
# ─────────────────────────────────────────────────────────────────────────────────────────────────

# ── Option 3: Re-parse WT.pdf (needs LLAMA_CLOUD_API_KEY + WT.pdf uploaded) ─────────────────────
# if not os.path.exists(MD_PATH):
#     from llama_parse import LlamaParse
#     docs = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data('/content/WT.pdf')
#     with open(MD_PATH, 'w', encoding='utf-8') as f:
#         f.write('\n\n'.join(d.text for d in docs))
# ─────────────────────────────────────────────────────────────────────────────────────────────────

md_text = open(MD_PATH, encoding='utf-8').read()

# TODO: filter for staging/treatment sections and chunk (~5-10 chunks for cost control)
# keywords = ['stage', 'treatment', 'chemotherapy', 'regimen', 'vincristine', 'dactinomycin', 'doxorubicin', 'side effect', 'histology']
# splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
# all_chunks = splitter.split_text(md_text)
# chunks = [c for c in all_chunks if any(k in c.lower() for k in keywords)][:8]
# print(f'{len(chunks)} chunks selected')

## Cell 3 — Extract graph with LLM

For each text chunk you will call the LLM chain and get back a `GraphExtraction` object with `.entities` and `.relationships`.

Key design decisions to keep in mind:
- **`temperature=0`** — deterministic; same chunk → same entities every run
- **Canonical names** — prompt must ask for lowercase entity names so deduplication works (`'stage iii'` not `'Stage III'`)
- **Entity deduplication** — key by lowercased name; skip if already seen
- **Relationships** — not deduplicated; duplicate edges overwrite safely in a `DiGraph`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# TODO: For each chunk, use structured output to extract entities + relationships
# llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(GraphExtraction)
# Aggregate all extractions
#
# Hints:
# 1. Build a system prompt instructing the model to extract clinical entities and relationships
#    that match our schema (Disease, Stage, Drug, ...)
# 2. Loop over chunks, invoke llm with the chunk text, collect .entities and .relationships
# 3. Deduplicate entities by name (lowercased)

all_entities = {}      # name -> Entity
all_relationships = []  # list of Relationship

# llm = ...
# prompt = ...
# for ch in chunks:
#     ex: GraphExtraction = llm.invoke(...)
#     ...

print(f'Entities: {len(all_entities)}, Relationships: {len(all_relationships)}')

## Cell 4 — Build NetworkX graph

Assemble extracted entities and relationships into a `nx.DiGraph`.

- **Nodes** = entities with `type` and `description` attributes
- **Edges** = relationships with `type` (e.g. `TREATS`) and `description` attributes
- Guard: only add an edge if both `r.source` and `r.target` already exist as nodes (avoids phantom nodes from extraction errors)

**Top-degree** nodes are the most-connected clinical concepts — expect `wilms tumor`, `stage iii`, or `vincristine` to appear at the top.

In [ ]:
import networkx as nx
G = nx.DiGraph()

# TODO: Add nodes (entities) with attributes (type, description)
# TODO: Add edges (relationships) with type and description
# Print: number of nodes, edges, top-degree nodes
#
# Hints:
# - G.add_node(entity.name, type=entity.type, description=entity.description)
# - G.add_edge(rel.source, rel.target, type=rel.type, description=rel.description)
# - top = sorted(G.degree, key=lambda x: -x[1])[:10]

# print(f'Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')
# print('Top hubs:', ...)

## Cell 5 — Visualize with pyvis

[pyvis](https://pyvis.readthedocs.io) renders an interactive force-directed graph inside Colab.

- Color each node by entity type; add a tooltip showing the description
- Use `cdn_resources='in_line'` so the HTML works without a network request in Colab
- **Important:** use `display(HTML(net.generate_html()))` — `net.show()` is deprecated in pyvis ≥0.7.0 and will fail silently in Colab

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

# TODO: Create interactive graph visualization
# Color nodes by type (Disease=red, Drug=blue, etc.)
#
# Hints:
# color_map = {"Disease":"#e74c3c", "Drug":"#3498db", "Stage":"#9b59b6",
#              "SideEffect":"#f39c12", "Procedure":"#1abc9c",
#              "Histology":"#2ecc71", "Outcome":"#34495e"}
# net = Network(height='600px', width='100%', directed=True, cdn_resources='in_line')
# for n, data in G.nodes(data=True):
#     net.add_node(n, label=n, color=color_map.get(data.get('type'), '#888'),
#                  title=f"{data.get('type')}: {data.get('description','')}")
# for s, t, d in G.edges(data=True):
#     net.add_edge(s, t, label=d.get('type',''))
# display(HTML(net.generate_html()))   # use generate_html() — net.show() is deprecated in Colab

## Cell 6 — Query the graph (graph traversal)

**Question:** *"What drugs treat Stage III Wilms tumor and what are their side effects?"*

This is a **2-hop traversal**:
```
Stage III  ←[TREATS]―  Drug  ―[CAUSES]→  SideEffect
```

Hint: the LLM extractor is not always consistent about edge direction for `TREATS` — check **both** `G.in_edges(stage)` and `G.out_edges(stage)` to find all drugs.

In [ ]:
# TODO: Multi-hop query — "What drugs treat Stage III Wilms tumor and what are their side effects?"
# 1. Find Stage III node (try fuzzy match: any node whose lower-case name contains 'stage iii' or 'stage 3')
# 2. Traverse incoming TREATS edges (drug -TREATS-> stage) to find drugs
# 3. Traverse CAUSES edges from drugs to find side effects (drug -CAUSES-> sideeffect)
# 4. Return structured answer (dict)

def multi_hop_query(G, stage_query='stage iii'):
    # find candidate stage node
    # find drugs treating it
    # find side effects caused by those drugs
    pass

# multi_hop_query(G)

## Cell 7 — Community detection (themes)

[Louvain community detection](https://en.wikipedia.org/wiki/Louvain_method) partitions the graph into clusters of densely-connected nodes. Each community is a *thematic module* — e.g., one cluster might group staging + treatment drugs; another might group toxicity entities.

Convert to undirected first (Louvain requires undirected), and pass `seed=42` for reproducibility.

In [ ]:
import networkx.algorithms.community as nx_comm

# TODO: Use Louvain or label propagation to find communities
# Each community is a "theme" — print members
#
# Hints:
# UG = G.to_undirected()
# communities = nx_comm.louvain_communities(UG, seed=42)  # or label_propagation_communities
# for i, c in enumerate(communities):
#     print(f'Community {i}: {sorted(c)}')

## Cell 8 — Compare with vector RAG

In [ ]:
# TODO: Same multi-hop question on vector RAG
# Show why vector struggles, why graph wins
#
# Hints:
# - Build a FAISS index from `chunks` with OpenAIEmbeddings
# - Retrieve top 4 for the same question
# - Pass to gpt-4o-mini and ask for a structured answer
# - Observe: vector retrieval may miss the cross-document link between Drug --CAUSES--> SideEffect

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# emb = OpenAIEmbeddings(model='text-embedding-3-small')
# vs = FAISS.from_texts(chunks, emb)
# question = 'What drugs treat Stage III Wilms tumor and what are their side effects?'
# docs = vs.similarity_search(question, k=4)
# ...

## Stretch — Natural-language to graph query

In [ ]:
# Stretch: Add a query function that uses the LLM to translate natural language into graph queries
#
# Idea: Have the LLM return a JSON traversal plan like
# {"start_node": "Stage III", "hops": [{"edge": "TREATS", "direction": "in"}, {"edge": "CAUSES", "direction": "out"}]}
# then execute that plan against G.

## KHCC connection

A KHCC oncology knowledge graph could link tumor boards, regimens, biomarkers, and outcomes — enabling multi-hop questions like *"Which Stage III favorable-histology patients on Regimen DD-4A had relapse and what salvage was used?"* — the kind of reasoning vector RAG cannot do alone.